# Review telomere insertion site plots

Interactively page through the zoomed-in insertion plots produced by
`src/visualize_telomere_insertions.py`. That script only ever writes a PDF per site
(`{prefix}{pid}_{chrom}_{center}.pdf` — the PNG output line is commented out/dead), at a
large 35x20-inch figure size, so this notebook rasterizes each PDF's first page and
downscales it for display. Mark each site Pass/Fail, add an optional note, and results
are written to a CSV as you go. A final cell aggregates the number of passed insertion
sites per PID.

Progress is resumable: re-running the notebook will skip plots already annotated in
the output CSV and resume where you left off.

In [ ]:
# One-time setup (uncomment and run if these aren't installed in your env)
# %pip install ipywidgets pymupdf pandas pillow

In [ ]:
from pathlib import Path

# --- CONFIGURE ME ---
RESULTS_DIR = Path("outputdir")

PLOT_DIR = RESULTS_DIR / "plots" / "zoomed_in"   # flat directory of {pid}_{chrom}_{center}.pdf files
CANDIDATE_REGION_TABLES_DIR = RESULTS_DIR / "candidate_region_tables"  # *_telomere_insertions_candidate_regions_extended_with_consensus.tsv per pid

ANNOTATIONS_CSV = RESULTS_DIR / "insertion_site_manual_annotations.csv"   # written to incrementally, one row per reviewed plot
SUMMARY_CSV = RESULTS_DIR / "insertion_site_summary_per_pid.csv"   # per-pid pass counts, written by the last cell
ANNOTATED_SITES_CSV = RESULTS_DIR / "insertion_sites_annotated_with_parameters.csv"   # full per-site table + annotation, written by the last cell

# visualize_telomere_insertions.py only ever writes .pdf (the .png savefig call is commented
# out in that script); .png/.jpg are supported here too in case that line ever gets re-enabled.
PLOT_EXTENSIONS = (".pdf", ".png", ".jpg", ".jpeg")

# The source figures are rendered at 35x20 inches, so keep RENDER_DPI modest and let
# DISPLAY_MAX_WIDTH downscale for on-screen review -- full res would be slow and huge.
RENDER_DPI = 100
DISPLAY_MAX_WIDTH = 1400

In [ ]:
import re
import csv
from datetime import datetime, timezone

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io

# Filenames follow "{pid}_{chrom}_{center}.pdf", e.g. "H021-ABCD_chr7_123456789.pdf".
# pid itself may contain underscores/hyphens (patient/sample IDs commonly do), so parse
# from the right: the last two underscore-separated tokens are chrom and center, and
# everything before that is the pid.
FILENAME_RE = re.compile(r"^(?P<pid>.+)_(?P<chrom>[^_]+)_(?P<center>\d+)$")


def parse_plot_filename(path: Path):
    stem = path.stem
    m = FILENAME_RE.match(stem)
    if not m:
        return {"pid": stem, "chrom": None, "center": None}
    return {"pid": m.group("pid"), "chrom": m.group("chrom"), "center": int(m.group("center"))}


# Rendering can be slow: these PDFs are vector plots that can contain thousands of
# per-read rectangles (the source script caps at <3000 reads per track, plotted at
# 35x20 inches), so MuPDF has a lot of vector geometry to rasterize. We render once per
# file and cache the PNG on disk (keyed by source mtime) so re-viewing/resuming is instant.
CACHE_DIR = Path(".plot_render_cache")
CACHE_DIR.mkdir(exist_ok=True)


def render_plot_to_image(path: Path, dpi: int = RENDER_DPI, max_width: int = DISPLAY_MAX_WIDTH) -> Image.Image:
    cache_path = CACHE_DIR / f"{path.stem}_{int(path.stat().st_mtime)}_{dpi}.png"
    if cache_path.exists():
        return Image.open(cache_path)

    if path.suffix.lower() == ".pdf":
        import fitz  # PyMuPDF
        doc = fitz.open(path)
        page = doc[0]
        zoom = dpi / 72
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        doc.close()
    else:
        img = Image.open(path)

    if max_width and img.width > max_width:
        ratio = max_width / img.width
        img = img.resize((max_width, int(img.height * ratio)), Image.LANCZOS)

    img.save(cache_path)
    return img

In [ ]:
# Discover plots and figure out what's already been annotated (for resuming)
all_plots = sorted(
    p for p in PLOT_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in PLOT_EXTENSIONS
)

CSV_FIELDS = ["pid", "chrom", "center", "filename", "decision", "note", "timestamp"]

if ANNOTATIONS_CSV.exists():
    existing = pd.read_csv(ANNOTATIONS_CSV)
    already_done = set(existing["filename"])
else:
    ANNOTATIONS_CSV.write_text(",".join(CSV_FIELDS) + "\n")
    already_done = set()

todo_plots = [p for p in all_plots if p.name not in already_done]

print(f"{len(all_plots)} plots found in {PLOT_DIR}")
print(f"{len(already_done)} already annotated in {ANNOTATIONS_CSV}")
print(f"{len(todo_plots)} remaining to review")

In [ ]:
import threading

def append_annotation(path: Path, decision: str, note: str):
    meta = parse_plot_filename(path)
    row = {
        "pid": meta["pid"],
        "chrom": meta["chrom"],
        "center": meta["center"],
        "filename": path.name,
        "decision": decision,
        "note": note,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    with open(ANNOTATIONS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writerow(row)


# --- Review UI state ---
state = {"index": 0}
_prefetch_cache = {}   # index -> rendered Image, filled by a background thread


def _prefetch(index):
    if 0 <= index < len(todo_plots) and index not in _prefetch_cache:
        try:
            _prefetch_cache[index] = render_plot_to_image(todo_plots[index])
        except Exception:
            pass  # surfaced again (and reported) when show_current() renders it directly


image_out = widgets.Output()
status_label = widgets.Label()
note_box = widgets.Text(placeholder="optional note / name", description="Note:", layout=widgets.Layout(width="500px"))
pass_btn = widgets.Button(description="Pass", button_style="success")
fail_btn = widgets.Button(description="Fail", button_style="danger")
skip_btn = widgets.Button(description="Skip (no annotation)")
prev_btn = widgets.Button(description="< Prev")


def show_current():
    image_out.clear_output(wait=True)
    note_box.value = ""
    if state["index"] >= len(todo_plots):
        status_label.value = "All plots reviewed."
        with image_out:
            print("Nothing left to review.")
        return

    idx = state["index"]
    path = todo_plots[idx]
    status_label.value = f"{idx + 1} / {len(todo_plots)}  —  {path.name}  (rendering...)"
    with image_out:
        print(f"Rendering {path.name} ...")  # immediate feedback -- first render of a file can take a while

    try:
        img = _prefetch_cache.pop(idx, None) or render_plot_to_image(path)
        image_out.clear_output(wait=True)
        status_label.value = f"{idx + 1} / {len(todo_plots)}  —  {path.name}"
        with image_out:
            display(img)
    except Exception as e:
        image_out.clear_output(wait=True)
        with image_out:
            print(f"Could not render {path}: {e}")

    # kick off rendering the next plot in the background so it's ready by the time we get there
    threading.Thread(target=_prefetch, args=(idx + 1,), daemon=True).start()


def go_next(_=None):
    state["index"] += 1
    show_current()


def on_pass(_):
    append_annotation(todo_plots[state["index"]], "pass", note_box.value)
    go_next()


def on_fail(_):
    append_annotation(todo_plots[state["index"]], "fail", note_box.value)
    go_next()


def on_skip(_):
    go_next()


def on_prev(_):
    if state["index"] > 0:
        state["index"] -= 1
        show_current()


pass_btn.on_click(on_pass)
fail_btn.on_click(on_fail)
skip_btn.on_click(on_skip)
prev_btn.on_click(on_prev)

controls = widgets.HBox([prev_btn, pass_btn, fail_btn, skip_btn])
display(widgets.VBox([status_label, image_out, note_box, controls]))
show_current()

## Aggregate: join annotations with candidate-region parameters, summarize per PID

Loads every `*_telomere_insertions_candidate_regions_extended_with_consensus.tsv` in
`CANDIDATE_REGION_TABLES_DIR` (one per PID, tab-separated), and left-joins your manual
annotations onto it by `(PID, chrom, insertion_site)` — the same `(pid, chrom, center)`
key used to name the plot files (`make_bed_for_visualization.R` sets the plot's
`region_center` to `insertion_site` directly, so this is an exact match, not a
binned/rounded one). The manual decision/note become new columns (`manual_decision`,
`manual_note`) alongside all of the pipeline's existing filter/ratio/consensus columns,
so you can see the parameters behind each Pass/Fail call.

Run this cell any time (including mid-review) — it always reflects whatever has been
annotated so far.

In [ ]:
candidate_files = sorted(
    CANDIDATE_REGION_TABLES_DIR.glob("*_telomere_insertions_candidate_regions_extended_with_consensus.tsv")
)
if not candidate_files:
    raise FileNotFoundError(f"No candidate region tables found in {CANDIDATE_REGION_TABLES_DIR}")

candidates = pd.concat((pd.read_csv(f, sep="\t") for f in candidate_files), ignore_index=True)

# region_center in the plot filename == insertion_site (see make_bed_for_visualization.R),
# falling back to the chromStart/chromEnd midpoint only where insertion_site is NA.
candidates["plot_center"] = candidates["insertion_site"]
missing = candidates["plot_center"].isna()
candidates.loc[missing, "plot_center"] = (
    (candidates.loc[missing, "chromStart"] + candidates.loc[missing, "chromEnd"]) / 2
)
candidates["plot_center"] = candidates["plot_center"].round().astype("Int64")

annotations = pd.read_csv(ANNOTATIONS_CSV)
annotations = annotations.rename(columns={"decision": "manual_decision", "note": "manual_note"})
annotations["center"] = annotations["center"].astype("Int64")

merged = candidates.merge(
    annotations[["pid", "chrom", "center", "manual_decision", "manual_note", "timestamp"]],
    left_on=["PID", "chrom", "plot_center"],
    right_on=["pid", "chrom", "center"],
    how="left",
).drop(columns=["pid", "center"])

merged.to_csv(ANNOTATED_SITES_CSV, index=False)
print(f"Wrote {ANNOTATED_SITES_CSV}: {len(merged)} candidate sites, {merged['manual_decision'].notna().sum()} annotated so far")

# reindex over every PID seen in the candidate tables (not just those with >=1 pass)
# so PIDs with zero passed insertions show up as 0 instead of being dropped.
summary = (
    merged[merged["manual_decision"] == "pass"]
    .groupby("PID")
    .size()
    .reindex(sorted(candidates["PID"].unique()), fill_value=0)
    .rename("n_passed_insertion_sites")
    .reset_index()
    .rename(columns={"index": "PID"})
)
summary.to_csv(SUMMARY_CSV, index=False)
print(f"Wrote {SUMMARY_CSV}")
summary